# AuraSync AS Dataset —  Three-Class DS-CNN + INT8 Live Test


In [ ]:
# 1. Install/import dependencies and configure deterministic execution.
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tensorflow-hub>=0.16.1', 'librosa>=0.10.2', 'soundfile>=0.12.1',
    'scikit-learn>=1.3', 'pandas>=2.0', 'matplotlib>=3.7',
    'seaborn>=0.13', 'tqdm>=4.66'
])

import os, io, re, gc, json, math, time, base64, random, shutil, hashlib, zipfile, wave
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import tensorflow as tf
import tensorflow_hub as hub
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from IPython.display import display, Audio

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


Python: 3.12.13
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 2. Project configuration.
PROJECT_NAME = 'AuraSync_AS_Final_3Class_HighAccuracy_INT8'
MODEL_VERSION = '2.0.0'
REPO_URL = 'https://github.com/GhimirePlan/AS_Final_Dataset_.git'
REPO_DIR = Path('/content/AS_Final_Dataset_')
WORK_DIR = Path('/content/aurasync_as_final_work')
DATASET_DIR = WORK_DIR / 'dataset_from_archives'
CACHE_DIR = WORK_DIR / 'cache'
EXPORT_DIR = WORK_DIR / 'export'

SEED = 20260713
SAMPLE_RATE = 16000
TARGET_SAMPLES = 16000
N_FFT = 512
WIN_LENGTH = 512
HOP_LENGTH = 320
N_MELS = 64
TIME_FRAMES = 49
FMIN = 80.0
FMAX = 7600.0
LOG_EPS = 1e-6

LABELS = ['stationary_background', 'speech_dominant', 'non_speech_activity']
LABEL_TO_ID = {name: i for i, name in enumerate(LABELS)}
ID_TO_LABEL = {i: name for name, i in LABEL_TO_ID.items()}
CLASS_COUNT = len(LABELS)

# Training strength and deployment acceptance gates.
STUDENT_SEEDS = [SEED, 1729]
EPOCHS = 70
BATCH_SIZE = 64
DISTILL_TEMPERATURE = 2.0
HARD_LOSS_WEIGHT = 0.78
DISTILL_LOSS_WEIGHT = 0.22 * (DISTILL_TEMPERATURE ** 2)
TARGET_TEST_ACCURACY = 0.85
TARGET_TEST_MACRO_F1 = 0.82
MAX_INT8_ACCURACY_DROP = 0.02

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

for directory in [WORK_DIR, CACHE_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Labels:', LABEL_TO_ID)
print('Work directory:', WORK_DIR)


Labels: {'stationary_background': 0, 'speech_dominant': 1, 'non_speech_activity': 2}
Work directory: /content/aurasync_as_final_work


## Dataset repository audit decision

The repository stores both extracted WAV files and ZIP archives. The loose tree is incomplete for `non_speech_activity`, while the training ZIP parts contain the full training class. This notebook performs a partial Git clone that checks out only ZIP files, validates every archive, and extracts the split/class information from the internal archive paths.

This avoids:

- training on duplicate loose files plus ZIP contents;
- losing 1,145 training clips from the incomplete loose folder;
- path-keyword guessing;
- accidental random split leakage.


In [ ]:
# 3. Clone only repository ZIP blobs (faster and avoids duplicate loose WAV checkout).
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_cmd = [
    'git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
    REPO_URL, str(REPO_DIR)
]
try:
    subprocess.run(clone_cmd, check=True)
except subprocess.CalledProcessError:
    print('Partial clone was not supported; falling back to shallow clone.')
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

repo_commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
tree_paths = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'ls-tree', '-r', '--name-only', 'HEAD'], text=True
).splitlines()
archive_paths = sorted(p for p in tree_paths if p.lower().endswith('.zip'))
if not archive_paths:
    raise RuntimeError('No ZIP archives found in the repository.')

# In a blobless/no-checkout clone, fetch exactly the required archives.
missing_archives = [p for p in archive_paths if not (REPO_DIR / p).is_file()]
if missing_archives:
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'checkout', 'HEAD', '--', *missing_archives],
        check=True
    )

print('Repository commit:', repo_commit)
print('ZIP archives:', len(archive_paths))
for p in archive_paths:
    print(' -', p)


Repository commit: f68c273f7ffdf79bb8750ff54e6c7280790b3d01
ZIP archives: 15
 - Test/AuraSync_data_test_non_speech_activity_part01.zip
 - Test/AuraSync_data_test_speech_dominant_part01.zip
 - Test/AuraSync_data_test_stationary_background_part01.zip
 - Train/Non-speech activity/AuraSync_data_train_non_speech_activity_part01.zip
 - Train/Non-speech activity/AuraSync_data_train_non_speech_activity_part02.zip
 - Train/Non-speech activity/AuraSync_data_train_non_speech_activity_part03.zip
 - Train/Non-speech activity/AuraSync_data_train_non_speech_activity_part04.zip
 - Train/Speech dominant/AuraSync_data_train_speech_dominant_part01.zip
 - Train/Speech dominant/AuraSync_data_train_speech_dominant_part02.zip
 - Train/Speech dominant/AuraSync_data_train_speech_dominant_part03.zip
 - Train/Stationary background/AuraSync_data_train_stationary_background_part01.zip
 - Train/Stationary background/AuraSync_data_train_stationary_background_part02.zip
 - Validation/AuraSync_data_validation_non_spee

In [ ]:
# 4. Safely extract authoritative WAV members and perform an exhaustive integrity audit.
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

def normalize_class_name(name):
    return re.sub(r'_+', '_', str(name).strip().lower().replace('-', '_').replace(' ', '_'))

def parse_archive_member(member_name):
    parts = list(PurePosixPath(member_name).parts)
    lower = [p.lower() for p in parts]
    if 'data' not in lower:
        return None
    i = lower.index('data')
    if len(parts) <= i + 3:
        return None
    split = parts[i + 1].lower()
    class_name = normalize_class_name(parts[i + 2])
    filename = parts[-1]
    if split not in {'train', 'validation', 'test'}:
        return None
    if class_name not in LABEL_TO_ID or not filename.lower().endswith('.wav'):
        return None
    return split, class_name, filename

rows = []
seen_ids = {}
archive_summary = []
for relative_archive in archive_paths:
    archive = REPO_DIR / relative_archive
    with zipfile.ZipFile(archive) as zf:
        bad_member = zf.testzip()
        if bad_member:
            raise RuntimeError(f'CRC failure in {relative_archive}: {bad_member}')
        accepted_in_archive = 0
        for info in zf.infolist():
            parsed = parse_archive_member(info.filename)
            if parsed is None:
                continue
            split, class_name, filename = parsed
            final_id = Path(filename).stem
            raw_bytes = zf.read(info)
            file_sha = hashlib.sha256(raw_bytes).hexdigest()
            with wave.open(io.BytesIO(raw_bytes), 'rb') as wav:
                params = (
                    wav.getnchannels(), wav.getsampwidth(),
                    wav.getframerate(), wav.getnframes()
                )
                pcm = wav.readframes(wav.getnframes())
            if params != (1, 2, SAMPLE_RATE, TARGET_SAMPLES):
                raise RuntimeError(f'Invalid WAV parameters {params}: {info.filename}')
            pcm_sha = hashlib.sha256(pcm).hexdigest()

            if final_id in seen_ids:
                prior = seen_ids[final_id]
                if prior != (split, class_name, pcm_sha):
                    raise RuntimeError(f'Conflicting duplicate ID: {final_id}')
                continue

            destination = DATASET_DIR / split / class_name / filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(raw_bytes)
            seen_ids[final_id] = (split, class_name, pcm_sha)
            rows.append({
                'final_id': final_id,
                'split': split,
                'class_name': class_name,
                'class_id': LABEL_TO_ID[class_name],
                'relative_path': destination.relative_to(DATASET_DIR).as_posix(),
                'source_archive': relative_archive,
                'wav_sha256': file_sha,
                'pcm_sha256': pcm_sha,
                'sample_rate': SAMPLE_RATE,
                'channels': 1,
                'frames': TARGET_SAMPLES,
            })
            accepted_in_archive += 1
        archive_summary.append({
            'archive': relative_archive,
            'wav_members_added': accepted_in_archive,
            'size_mb': archive.stat().st_size / (1024 ** 2),
        })

manifest = pd.DataFrame(rows).sort_values(['split', 'class_name', 'final_id']).reset_index(drop=True)
manifest.to_csv(WORK_DIR / 'repository_dataset_manifest.csv', index=False)
pd.DataFrame(archive_summary).to_csv(WORK_DIR / 'archive_audit.csv', index=False)

if manifest['final_id'].duplicated().any() or manifest['pcm_sha256'].duplicated().any():
    raise RuntimeError('Duplicate final IDs or PCM content remain after extraction.')

counts = manifest.groupby(['split', 'class_name']).size().unstack(fill_value=0)
print('Extracted clips:', len(manifest))
display(counts)
display(pd.DataFrame(archive_summary))

# Lock the counts observed in the audited repository snapshot. A later repository update
# is disclosed as version drift instead of silently changing the experiment population.
current_snapshot_reference = {
    ('train', 'stationary_background'): 1936,
    ('train', 'speech_dominant'): 2388,
    ('train', 'non_speech_activity'): 4426,
    ('validation', 'stationary_background'): 415,
    ('validation', 'speech_dominant'): 512,
    ('validation', 'non_speech_activity'): 913,
    ('test', 'stationary_background'): 415,
    ('test', 'speech_dominant'): 512,
    ('test', 'non_speech_activity'): 949,
}
snapshot_drift = []
for (split, class_name), expected in current_snapshot_reference.items():
    actual = int(counts.loc[split, class_name]) if split in counts.index and class_name in counts.columns else 0
    if actual != expected:
        snapshot_drift.append({'split': split, 'class_name': class_name, 'audited_snapshot': expected, 'current': actual, 'difference': actual - expected})
if snapshot_drift:
    print('REPOSITORY VERSION-DRIFT WARNING:')
    display(pd.DataFrame(snapshot_drift))

# The earlier verified manifest had 949 validation non-speech clips. The current repository
# is short by 36; disclose the gap without moving any train/test clips into validation.
prior_verified_reference = {
    **current_snapshot_reference,
    ('validation', 'non_speech_activity'): 949,
}
gaps = []
for (split, class_name), expected in prior_verified_reference.items():
    actual = int(counts.loc[split, class_name]) if split in counts.index and class_name in counts.columns else 0
    if actual != expected:
        gaps.append({'split': split, 'class_name': class_name, 'expected': expected, 'actual': actual, 'difference': actual - expected})
if gaps:
    print('REPOSITORY COMPLETENESS WARNING:')
    display(pd.DataFrame(gaps))

for split in ['train', 'validation', 'test']:
    for class_name in LABELS:
        n = int(((manifest['split'] == split) & (manifest['class_name'] == class_name)).sum())
        if n < 300:
            raise RuntimeError(f'Too few clips for {split}/{class_name}: {n}')


Extracted clips: 12466


class_name,non_speech_activity,speech_dominant,stationary_background
split,,,
test,949,512,415
train,4426,2388,1936
validation,913,512,415


,archive,wav_members_added,size_mb
0,Test/AuraSync_data_test_non_speech_activity_pa...,949,25.362008
1,Test/AuraSync_data_test_speech_dominant_part01...,512,11.940896
2,Test/AuraSync_data_test_stationary_background_...,415,10.784045
3,Train/Non-speech activity/AuraSync_data_train_...,1145,30.416145
4,Train/Non-speech activity/AuraSync_data_train_...,1145,30.285066
5,Train/Non-speech activity/AuraSync_data_train_...,1145,30.564631
6,Train/Non-speech activity/AuraSync_data_train_...,991,26.364442
7,Train/Speech dominant/AuraSync_data_train_spee...,1145,26.883219
8,Train/Speech dominant/AuraSync_data_train_spee...,1145,27.052396
9,Train/Speech dominant/AuraSync_data_train_spee...,98,2.240901


REPOSITORY COMPLETENESS WARNING:


,split,class_name,expected,actual,difference
0,validation,non_speech_activity,949,913,-36
